# ZerO: ResNet-18 / ResNet-50 и обучение

**Смысл статьи.** Случайность начальных весов можно заменить структурой: identity сохраняет сигнал, а преобразование Адамара при расширении каналов помогает избежать вырождения признаков. Последняя свёртка остаточной ветви получает нули; остальные — структурированные веса. Это не полное обнуление сети и не обнуление γ BatchNorm.

Для матрицы $W\in\mathbb{R}^{p\times q}$:
- при $p\le q$: частичная единичная матрица;
- при $p>q$: $W=\sqrt{2/N}\,H_N[:p,:q]$, $N=2^{\lceil\log_2 p\rceil}$.

У свёрток заполняется только центр ядра. $H_N$ содержит **+1 и −1**, поэтому веса включают отрицательные значения с масштабным коэффициентом, несмотря на название статьи. BatchNorm: γ=1, β=0.

Авторы также изучают траектории ранга и разброс качества между запусками. Детерминированные веса не устраняют случайность порядка батчей и аугментаций.

Источник: [Zhao et al., Algorithms 1–2, Sections 3–5](https://arxiv.org/pdf/2110.12661).

Здесь — учебный эксперимент на CIFAR-10 с BatchNorm, а не полное воспроизведение всех результатов статьи. Для обеих сетей используем stem 3×3 без начального pooling, чтобы сохранить разрешение 32×32. ImageNet, обучение без BN и pruning в этот эксперимент не входят.

In [1]:
%pip install torch torchvision numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import torch.nn as nn

In [2]:
@torch.no_grad()
def hadamard_matrix(size, *, device=None, dtype=torch.float32):
    if size < 1 or size & (size - 1):
        raise ValueError("Размер Hadamard должен быть степенью двойки")
    h = torch.ones(1, 1, device=device, dtype=dtype)
    while h.shape[0] < size:
        h = torch.cat((torch.cat((h, h), dim=1),
                       torch.cat((h, -h), dim=1)), dim=0)
    return h


@torch.no_grad()
def structured_init_(weight, use_hadamard=True):
    if weight.ndim not in (2, 4):
        raise ValueError("Ожидается Linear [out, in] или Conv2d [out, in, kH, kW]")
    if weight.ndim == 4 and any(k % 2 == 0 for k in weight.shape[2:]):
        raise ValueError("Для центральной инициализации нужны нечётные размеры ядра")
    rows, cols = weight.shape[:2]
    if use_hadamard and rows > cols:
        size = 1 << (rows - 1).bit_length()
        matrix = hadamard_matrix(size, device=weight.device, dtype=weight.dtype)
        # Нормировка Algorithm 1 статьи: sqrt(2/N).
        matrix = matrix[:rows, :cols] * (2.0 / size) ** 0.5
    else:
        matrix = torch.eye(rows, cols, device=weight.device, dtype=weight.dtype)
    weight.zero_()
    if weight.ndim == 2:
        weight.copy_(matrix)
    else:
        weight[:, :, weight.shape[2] // 2, weight.shape[3] // 2].copy_(matrix)
    return weight


def zero_init_(weight):
    return structured_init_(weight, use_hadamard=True)


def identity_init_(weight):
    # Ablation: убираем Hadamard, сохраняя остальную схему ZerO.
    return structured_init_(weight, use_hadamard=False)

В [демонстрационном MNIST-ноутбуке авторов](https://github.com/jiaweizzhao/ZerO-initialization/blob/main/example_mnist.ipynb) используется $1/\sqrt{N}$; здесь явно выбрана формула $\sqrt{2/N}$ из Algorithm 1 PDF. Эти варианты отличаются множителем $\sqrt{2}$, их не следует незаметно смешивать.

Режимы: `init="zero"` — ZerO; `"identity"` — ablation без Hadamard; `"zeros"` — все Conv/Linear нулевые (отрицательный контроль); `"kaiming"`, `"xavier"`, `"normal"` или функция `init(weight)` — прежний интерфейс. `zero_init_residual=True` остаётся отдельной опцией для случайных инициализаций.

In [3]:
class ResBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        expanded_channels = out_channels * self.expansion
        self.conv1 = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, expanded_channels, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(expanded_channels)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != expanded_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, expanded_channels, 1, stride, bias=False),
                nn.BatchNorm2d(expanded_channels),
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        return self.relu(out + identity)

In [4]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000, in_channels=3,
                 init="kaiming", zero_init_residual=False, cifar_stem=False):
        super().__init__()
        self._channels = 64
        self.conv1 = (nn.Conv2d(in_channels, 64, 3, 1, 1, bias=False) if cifar_stem
                      else nn.Conv2d(in_channels, 64, 7, 2, 3, bias=False))
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.Identity() if cifar_stem else nn.MaxPool2d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)
        self.initialize_weights(init, zero_init_residual)

    def _make_layer(self, block, channels, count, stride=1):
        layers = [block(self._channels, channels, stride)]
        self._channels = channels * block.expansion
        layers.extend(block(self._channels, channels) for _ in range(1, count))
        return nn.Sequential(*layers)

    @torch.no_grad()
    def initialize_weights(self, init="kaiming", zero_init_residual=False):
        """Инициализировать Conv2d/Linear выбранным методом, BatchNorm — отдельно.

        init: 'zero', 'identity', 'kaiming', 'xavier', 'normal', 'zeros' или функция init(weight),
        изменяющая переданный тензор на месте. Bias обнуляются.
        Метод можно вызывать повторно для нового эксперимента.
        """
        if isinstance(init, str) and init in {"zero", "identity"} and zero_init_residual:
            raise ValueError("Для ZerO/identity оставьте zero_init_residual=False: обнуляется последняя Conv, а не BN")
        initializers = {
            "zero": zero_init_,
            "identity": identity_init_,
            "kaiming": lambda w: nn.init.kaiming_normal_(w, mode="fan_out", nonlinearity="relu"),
            "xavier": nn.init.xavier_normal_,
            "normal": lambda w: nn.init.normal_(w, mean=0.0, std=0.01),
            "zeros": nn.init.zeros_,
        }
        if isinstance(init, str):
            if init not in initializers:
                raise ValueError(f"Неизвестная инициализация {init!r}. Варианты: {list(initializers)}")
            initializer = initializers[init]
        elif callable(init):
            initializer = init
        else:
            raise TypeError("init должен быть строкой или функцией init(weight)")

        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                # Для линейного выхода в режиме Kaiming используем gain=1.
                if isinstance(init, str) and init == "kaiming" and isinstance(module, nn.Linear):
                    nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="linear")
                else:
                    initializer(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.BatchNorm2d):
                module.reset_running_stats()
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

        # Алгоритм ResNet из статьи: последняя свёртка нулевая, gamma BN = 1.
        if isinstance(init, str) and init in {"zero", "identity"}:
            for module in self.modules():
                if isinstance(module, ResBlock):
                    nn.init.zeros_(module.conv2.weight)
                elif isinstance(module, Bottleneck):
                    nn.init.zeros_(module.conv3.weight)

        if zero_init_residual:
            for module in self.modules():
                if isinstance(module, ResBlock):
                    nn.init.zeros_(module.bn2.weight)
                elif isinstance(module, Bottleneck):
                    nn.init.zeros_(module.bn3.weight)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = torch.flatten(self.avgpool(x), 1)
        return self.fc(x)


class ResNet18(ResNet):
    def __init__(self, num_classes=1000, in_channels=3, init="kaiming", zero_init_residual=False, cifar_stem=False):
        super().__init__(ResBlock, [2, 2, 2, 2], num_classes, in_channels, init, zero_init_residual, cifar_stem)


class ResNet50(ResNet):
    def __init__(self, num_classes=1000, in_channels=3, init="kaiming", zero_init_residual=False, cifar_stem=False):
        super().__init__(Bottleneck, [3, 4, 6, 3], num_classes, in_channels, init, zero_init_residual, cifar_stem)

## Проверка идеи до обучения

Первый тест проверяет независимость начальных весов от seed. Второй показывает, что после обновления нулевая последняя свёртка остаточной ветви может начать обучаться; все её предшественники не обязаны иметь ненулевой градиент на самом первом шаге.

In [5]:
def check_zero_initialization():
    for model_cls in (ResNet18, ResNet50):
        torch.manual_seed(1)
        first = model_cls(num_classes=10, init="zero", cifar_stem=True)
        torch.manual_seed(99)
        second = model_cls(num_classes=10, init="zero", cifar_stem=True)
        assert all(torch.equal(a, b) for a, b in zip(first.state_dict().values(), second.state_dict().values()))
        for block in first.modules():
            if isinstance(block, (ResBlock, Bottleneck)):
                last_conv = block.conv2 if isinstance(block, ResBlock) else block.conv3
                last_bn = block.bn2 if isinstance(block, ResBlock) else block.bn3
                assert torch.count_nonzero(last_conv.weight) == 0
                assert torch.all(last_bn.weight == 1)
        del second
        first.train()
        optimizer = torch.optim.SGD(first.parameters(), lr=0.01)
        images, labels = torch.randn(2, 3, 32, 32), torch.tensor([0, 1])
        target = first.layer4[-1]
        weight = target.conv2.weight if isinstance(target, ResBlock) else target.conv3.weight
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.cross_entropy(first(images), labels)
        loss.backward()
        assert weight.grad is not None and torch.isfinite(weight.grad).all()
        assert weight.grad.abs().sum() > 0
        optimizer.step()
        assert weight.abs().sum() > 0
        print(f"{model_cls.__name__}: deterministic initialization / gradient / update OK")


check_zero_initialization()

ResNet18: deterministic initialization / gradient / update OK
ResNet50: deterministic initialization / gradient / update OK


## Конфигурация эксперимента

`smoke` — 1 эпоха, 512 train / 256 val / 256 test изображений, один seed и ResNet-18: проверка работоспособности, не оценка качества метода. `full` — обе сети, три seed, 200 эпох; это длительный запуск. Для десяти повторов задайте `seeds=tuple(range(10))`.

SGD: lr=0.1, momentum=0.9, weight decay=1e-4; warmup 10 эпох для CIFAR-10 взят из [раздела 4 статьи](https://arxiv.org/pdf/2110.12661). Cosine decay, 200 эпох и отдельная validation-выборка — выбор данного ноутбука. Одинаковые seed, split, порядок данных, аугментации и расписание используются для всех методов.

Validation (5 000 изображений из train) выбирает checkpoint; test используется только после обучения. Для сопоставления с таблицами статьи потребуется согласовать полный протокол, а не сравнивать числа напрямую.

In [6]:
import csv
import json
import math
import random
import time
from dataclasses import dataclass, asdict, replace
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torchvision
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


@dataclass
class ExperimentConfig:
    architectures: tuple = ("resnet18", "resnet50")
    methods: tuple = ("kaiming", "xavier", "zero")
    seeds: tuple = (0, 1, 2)
    epochs: int = 200
    warmup_epochs: int = 10
    batch_size: int = 128
    lr: float = 0.1
    momentum: float = 0.9
    weight_decay: float = 1e-4
    val_size: int = 5000
    split_seed: int = 2026
    train_limit: int | None = None
    eval_limit: int | None = None
    num_workers: int = 0  # Работает в notebook на macOS без multiprocessing spawn.
    rank_every: int = 10
    data_dir: str = "data"
    output_dir: str = "runs/zero-init"
    device: str = "auto"


PRESET = "smoke"  # Поменять на "full" для длительного сравнения.
full_config = ExperimentConfig()
smoke_config = replace(full_config, architectures=("resnet18",), seeds=(0,),
                       epochs=1, warmup_epochs=1, batch_size=32,
                       train_limit=512, eval_limit=256, rank_every=1)
config = {"smoke": smoke_config, "full": full_config}[PRESET]
# Дополнительные ablation-эксперименты:
# config = replace(config, methods=("kaiming", "zero", "identity", "zeros"))
# config = replace(config, architectures=("resnet50",))


def select_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


print(PRESET, asdict(config))
print("Device:", select_device(config.device))

smoke {'architectures': ('resnet18',), 'methods': ('kaiming', 'xavier', 'zero'), 'seeds': (0,), 'epochs': 1, 'warmup_epochs': 1, 'batch_size': 32, 'lr': 0.1, 'momentum': 0.9, 'weight_decay': 0.0001, 'val_size': 5000, 'split_seed': 2026, 'train_limit': 512, 'eval_limit': 256, 'num_workers': 0, 'rank_every': 1, 'data_dir': 'data', 'output_dir': 'runs/zero-init', 'device': 'auto'}
Device: mps


## CIFAR-10

Данные скачиваются при первом запуске `run_experiments`. Train: случайный crop с padding=4 и horizontal flip; validation/test: только нормализация. Подмножества и validation split фиксируются отдельно от seed обучения.

In [7]:
def make_loaders(cfg, seed):
    normalize = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), normalize,
    ])
    eval_transform = transforms.Compose([transforms.ToTensor(), normalize])
    train_data = datasets.CIFAR10(cfg.data_dir, train=True, download=True, transform=train_transform)
    val_data = datasets.CIFAR10(cfg.data_dir, train=True, download=False, transform=eval_transform)
    test_data = datasets.CIFAR10(cfg.data_dir, train=False, download=True, transform=eval_transform)
    if not 0 < cfg.val_size < len(train_data):
        raise ValueError("val_size должен быть между 1 и размером train - 1")
    generator = torch.Generator().manual_seed(cfg.split_seed)
    indices = torch.randperm(len(train_data), generator=generator).tolist()
    val_indices, train_indices = indices[:cfg.val_size], indices[cfg.val_size:]
    if cfg.train_limit is not None:
        train_indices = train_indices[:cfg.train_limit]
    if cfg.eval_limit is not None:
        val_indices = val_indices[:cfg.eval_limit]
    test_indices = torch.randperm(len(test_data), generator=generator).tolist()
    if cfg.eval_limit is not None:
        test_indices = test_indices[:cfg.eval_limit]
    common = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers,
                  pin_memory=select_device(cfg.device).type == "cuda")
    train_loader = DataLoader(Subset(train_data, train_indices), shuffle=True,
                              generator=torch.Generator().manual_seed(seed), **common)
    val_loader = DataLoader(Subset(val_data, val_indices), shuffle=False,
                            generator=torch.Generator().manual_seed(seed + 1), **common)
    test_loader = DataLoader(Subset(test_data, test_indices), shuffle=False,
                             generator=torch.Generator().manual_seed(seed + 2), **common)
    return train_loader, val_loader, test_loader


def create_model(architecture, method):
    classes = {"resnet18": ResNet18, "resnet50": ResNet50}
    return classes[architecture](num_classes=10, init=method, cifar_stem=True)

## Метрики и цикл обучения

Stable rank измеряем у `layer2[0].conv1`, `layer3[0].conv1`, `layer4[0].conv1`: разворачиваем ядро в $[C_{out}, C_{in}k_Hk_W]$ и считаем $\|W\|_F^2/\sigma_{max}^2(W)$. Для нулевой матрицы принимаем 0. Это диагностическая метрика выбранной матричной развёртки, не ранг всей сети. SVD выполняется на CPU раз в `rank_every` эпох; частоту можно уменьшить для ускорения.

Сохраняются config, версии библиотек, история JSON/CSV, лучший по validation checkpoint и итоговая test-метрика. Каталог каждого запуска уникален. Фиксация seed помогает повторить эксперимент на той же платформе, но не гарантирует побитового совпадения CPU/CUDA/MPS.

In [8]:
@torch.no_grad()
def stable_ranks(model):
    result = {}
    for name in ("layer2", "layer3", "layer4"):
        w = getattr(model, name)[0].conv1.weight.detach().float().cpu().flatten(1)
        singular_values = torch.linalg.svdvals(w)
        spectral_sq = singular_values[0].square().item()
        result[name] = float(w.square().sum().item() / spectral_sq) if spectral_sq else 0.0
    return result


def learning_rate_at(step, total_steps, warmup_steps, peak_lr):
    if step < warmup_steps:
        return peak_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps - 1)
    return peak_lr * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def run_epoch(model, loader, device, optimizer=None, cfg=None, epoch=0):
    training = optimizer is not None
    model.train(training)
    loss_sum, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for batch_index, (images, labels) in enumerate(loader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if training:
                step = epoch * len(loader) + batch_index
                lr = learning_rate_at(step, cfg.epochs * len(loader),
                                      cfg.warmup_epochs * len(loader), cfg.lr)
                for group in optimizer.param_groups:
                    group["lr"] = lr
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = nn.functional.cross_entropy(logits, labels)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Loss не конечен: epoch={epoch + 1}, batch={batch_index}")
            if training:
                loss.backward()
                optimizer.step()
            batch_size = labels.size(0)
            loss_sum += loss.detach().item() * batch_size
            correct += (logits.argmax(1) == labels).sum().item()
            count += batch_size
    if count == 0:
        raise ValueError("Пустой DataLoader")
    return {"loss": loss_sum / count, "accuracy": 100.0 * correct / count}


def save_history(history, directory):
    (directory / "history.json").write_text(json.dumps(history, indent=2))
    with (directory / "history.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)


def train_one(cfg, architecture, method, seed, directory, loaders=None):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    if cfg.epochs < 1 or not 0 <= cfg.warmup_epochs <= cfg.epochs or cfg.rank_every < 1:
        raise ValueError("Нужны epochs>=1, 0<=warmup_epochs<=epochs, rank_every>=1")
    seed_everything(seed)
    train_loader, val_loader, test_loader = loaders if loaders is not None else make_loaders(cfg, seed)
    device = select_device(cfg.device)
    model = create_model(architecture, method).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr,
                                momentum=cfg.momentum, weight_decay=cfg.weight_decay)
    # Создание моделей потребляет RNG по-разному. Сбрасываем его перед аугментациями.
    seed_everything(seed)
    metadata = {**asdict(cfg), "architecture": architecture, "method": method,
                "seed": seed, "actual_device": str(device), "torch_version": str(torch.__version__),
                "torchvision_version": str(torchvision.__version__), "numpy_version": str(np.__version__),
                "sizes": {"train": len(train_loader.dataset), "val": len(val_loader.dataset),
                          "test": len(test_loader.dataset)}}
    (directory / "config.json").write_text(json.dumps(metadata, indent=2))
    initial_val = run_epoch(model, val_loader, device)
    history = [{"epoch": 0, "lr": 0.0, "train_loss": None, "train_accuracy": None,
                "val_loss": initial_val["loss"], "val_accuracy": initial_val["accuracy"],
                "seconds": 0.0, **{f"rank_{k}": v for k, v in stable_ranks(model).items()}}]
    save_history(history, directory)
    best_accuracy, best_epoch = -1.0, 0
    for epoch in range(cfg.epochs):
        started = time.perf_counter()
        train_metrics = run_epoch(model, train_loader, device, optimizer, cfg, epoch)
        val_metrics = run_epoch(model, val_loader, device)
        measure_rank = (epoch + 1) % cfg.rank_every == 0 or epoch + 1 == cfg.epochs
        ranks = stable_ranks(model) if measure_rank else dict.fromkeys(("layer2", "layer3", "layer4"))
        row = {"epoch": epoch + 1, "lr": optimizer.param_groups[0]["lr"],
               "train_loss": train_metrics["loss"], "train_accuracy": train_metrics["accuracy"],
               "val_loss": val_metrics["loss"], "val_accuracy": val_metrics["accuracy"],
               "seconds": time.perf_counter() - started,
               **{f"rank_{k}": v for k, v in ranks.items()}}
        history.append(row)
        save_history(history, directory)
        if val_metrics["accuracy"] > best_accuracy:
            best_accuracy, best_epoch = val_metrics["accuracy"], epoch + 1
            torch.save({"model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                        "epoch": best_epoch, "val_accuracy": best_accuracy,
                        "architecture": architecture, "method": method}, directory / "best.pt")
        print(f"{architecture}/{method}/seed={seed} epoch={epoch + 1}/{cfg.epochs} "
              f"loss={train_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.2f}% "
              f"lr={row['lr']:.5f} time={row['seconds']:.1f}s", flush=True)
    checkpoint = torch.load(directory / "best.pt", map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model_state"])
    test_metrics = run_epoch(model, test_loader, device)
    summary = {"architecture": architecture, "method": method, "seed": seed,
               "best_epoch": best_epoch, "best_val_accuracy": best_accuracy,
               "test_loss": test_metrics["loss"], "test_accuracy": test_metrics["accuracy"],
               "test_error": 100.0 - test_metrics["accuracy"], "directory": str(directory)}
    (directory / "summary.json").write_text(json.dumps(summary, indent=2))
    return summary


def run_experiments(cfg):
    root = Path(cfg.output_dir) / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    root.mkdir(parents=True, exist_ok=False)
    summaries = []
    print("Results:", root.resolve(), flush=True)
    for architecture in cfg.architectures:
        for seed in cfg.seeds:
            for method in cfg.methods:
                directory = root / f"{architecture}-{method}-seed{seed}"
                summaries.append(train_one(cfg, architecture, method, seed, directory))
                with (root / "summary.csv").open("w", newline="") as f:
                    writer = csv.DictWriter(f, fieldnames=list(summaries[0]))
                    writer.writeheader()
                    writer.writerows(summaries)
    return root, summaries

## Запуск

Ячейка ниже запускает выбранный `PRESET`. Перед длительным запуском выберите `full` в конфигурации и выполните её заново. Для smoke достаточно CPU; полное сравнение особенно ResNet-50 потребует существенно больше вычислений.

In [10]:
results_dir, summaries = run_experiments(config)

Results: /Users/ilya/Documents/compet/itmo/efficient-nn-hw/hw2/runs/zero-init/20261009-191600-043071


URLError: <urlopen error [Errno 8] nodename nor servname provided, or not known>

## Графики в стиле статьи

Основная фигура повторяет компоновку [Figure 5 (стр. 10)](https://arxiv.org/pdf/2110.12661#page=10): 2×3 панели stable rank. Верхний ряд — ResNet-18, нижний — ResNet-50; столбцы — первые свёртки групп 2, 3, 4. ZerO — синий, Kaiming — зелёный, Xavier — красный; светлый серо-сиреневый фон и белая сетка. Loss/accuracy вынесены в отдельные фигуры.

Кривые строятся из наших `history.json`, без сглаживания или подгонки к статье. В нашем эксперименте обе сети используют CIFAR-10; у авторов нижний ряд Figure 5 относится к ImageNet. Диапазоны осей определяются фактическими значениями. При `smoke` нижний ряд останется пустым: для него запустите ResNet-50 или `full`.

По умолчанию показаны средние линии без полос, как в статье; `show_std=True` добавляет ±1 выборочное стандартное отклонение. Среднее считается на общих записанных эпохах всех seed данного метода. Для более подробных траекторий перед обучением задайте `config = replace(config, rank_every=1)` (SVD станет чаще).

Сохраняются `figure5-stable-rank.png/.pdf` и отдельные `<architecture>-training-curves.png/.pdf`; PNG — 300 dpi, PDF — векторный. Графики доступны и для незавершённого запуска. Figure 3, 4 и 6 требуют отдельных экспериментов (MLP, глубина сети, pruning/low-rank), которых в текущем обучении нет.


In [ ]:
# Оформление Figure 5; значения берутся только из history.json.
PAPER_STYLE = {
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.facecolor": "#EAEAF2", "figure.facecolor": "white",
    "axes.edgecolor": "white", "axes.linewidth": 0.8,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "white", "grid.linewidth": 0.7,
    "xtick.color": "#262626", "ytick.color": "#262626",
    "xtick.major.size": 0, "ytick.major.size": 0,
    "legend.frameon": False, "legend.fontsize": 8,
    "lines.linewidth": 1.5, "pdf.fonttype": 42,
}
METHOD_STYLES = {
    "zero": ("ZerO Init", "#0072BD", "-"),
    "kaiming": ("Kaiming Init (Random)", "#00A000", "-"),
    "xavier": ("Xavier Init (Random)", "#FF0000", "-"),
    "identity": ("Identity Init", "#9467BD", "--"),
    "zeros": ("All-zero Init", "#555555", ":"),
    "normal": ("Normal Init (Random)", "#E69F00", "--"),
}


def load_plot_runs(root):
    # Работает и во время обучения, до появления summary.json.
    runs = []
    for history_path in sorted(Path(root).glob("*/history.json")):
        directory = history_path.parent
        cfg_path = directory / "config.json"
        if not cfg_path.exists():
            continue
        cfg = json.loads(cfg_path.read_text())
        summary_path = directory / "summary.json"
        runs.append({"config": cfg, "history": json.loads(history_path.read_text()),
                     "summary": json.loads(summary_path.read_text()) if summary_path.exists() else None})
    if not runs:
        raise ValueError(f"В {root} нет history.json с config.json. Сначала запустите обучение.")
    return runs


def plot_metric(ax, runs, metric, show_std=False):
    available = {r["config"]["method"] for r in runs}
    methods = [m for m in METHOD_STYLES if m in available] + sorted(available - METHOD_STYLES.keys())
    for method in methods:
        selected = [r for r in runs if r["config"]["method"] == method]
        by_epoch = []
        for run in selected:
            by_epoch.append({row["epoch"]: float(row[metric]) for row in run["history"]
                             if row.get(metric) is not None and np.isfinite(row[metric])})
        # Одинаковый набор seed в каждой точке; без интерполяции пропусков.
        epochs = sorted(set.intersection(*(set(values) for values in by_epoch)))
        if not epochs:
            continue
        samples = np.array([[values[e] for e in epochs] for values in by_epoch])
        mean = samples.mean(axis=0)
        label, color, linestyle = METHOD_STYLES.get(method, (method, "#8C564B", "--"))
        ax.plot(epochs, mean, color=color, linestyle=linestyle, label=label,
                marker="o" if len(epochs) <= 2 else None, markersize=3)
        if show_std and len(samples) > 1:
            std = samples.std(axis=0, ddof=1)
            ax.fill_between(epochs, mean - std, mean + std, color=color, alpha=0.13, linewidth=0)
    ax.margins(x=0.04)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
    ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=5, integer=True))


def save_figure(fig, root, name):
    for extension in ("png", "pdf"):
        fig.savefig(Path(root) / f"{name}.{extension}", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_rank_trajectories(root, runs=None, show_std=False):
    """Аналог Figure 5: строки — сети, столбцы — группы 2, 3, 4."""
    runs = load_plot_runs(root) if runs is None else runs
    architectures = ("resnet18", "resnet50")
    with plt.rc_context(PAPER_STYLE):
        fig, axes = plt.subplots(2, 3, figsize=(9.0, 5.2), squeeze=False)
        for row_index, architecture in enumerate(architectures):
            selected = [r for r in runs if r["config"]["architecture"] == architecture]
            for column, layer in enumerate(("layer2", "layer3", "layer4")):
                ax = axes[row_index, column]
                if selected:
                    plot_metric(ax, selected, f"rank_{layer}", show_std)
                else:
                    ax.text(0.5, 0.5, "No recorded run", ha="center", va="center", transform=ax.transAxes)
                    ax.set_xticks([])
                    ax.set_yticks([])
                if column == 0:
                    ax.set_ylabel("Stable Rank")
                if row_index == 1:
                    ax.set_xlabel("Epochs")
                if row_index == 0:
                    ax.set_title(f"Group {column + 2}", fontsize=10, pad=7)
            axes[row_index, 0].annotate(
                f"{architecture.replace('resnet', 'ResNet-')} / CIFAR-10",
                xy=(-0.34, 0.5), xycoords="axes fraction", rotation=90,
                ha="center", va="center", fontsize=9,
            )
        # Одна легенда, как в Figure 5; при отсутствии первой строки — вторая.
        for ax in axes[:, 0]:
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                ax.legend(handles, labels, loc="upper right")
                break
        fig.subplots_adjust(left=0.12, right=0.99, bottom=0.10, top=0.94, hspace=0.30, wspace=0.25)
        save_figure(fig, root, "figure5-stable-rank")


def plot_training_curves(root, runs, show_std=False):
    # Дополнительная диагностика отдельно от композиции Figure 5.
    with plt.rc_context(PAPER_STYLE):
        for architecture in sorted({r["config"]["architecture"] for r in runs}):
            selected = [r for r in runs if r["config"]["architecture"] == architecture]
            fig, axes = plt.subplots(1, 3, figsize=(9.0, 2.8))
            for ax, metric, label in zip(axes, ("train_loss", "val_accuracy", "train_accuracy"),
                                        ("Train Loss", "Validation Accuracy (%)", "Train Accuracy (%)")):
                plot_metric(ax, selected, metric, show_std)
                ax.set(xlabel="Epochs", ylabel=label)
            axes[0].legend(loc="best")
            fig.suptitle(f"{architecture.replace('resnet', 'ResNet-')} / CIFAR-10", fontsize=11)
            fig.tight_layout()
            save_figure(fig, root, f"{architecture}-training-curves")


def plot_results(root, show_std=False):
    root = Path(root)
    runs = load_plot_runs(root)
    plot_rank_trajectories(root, runs, show_std)
    plot_training_curves(root, runs, show_std)
    records = [r["summary"] for r in runs if r["summary"] is not None]
    aggregate = []
    for architecture, method in sorted({(r["architecture"], r["method"]) for r in records}):
        errors = [r["test_error"] for r in records
                  if r["architecture"] == architecture and r["method"] == method]
        aggregate.append({"architecture": architecture, "method": method, "runs": len(errors),
                          "mean_test_error": float(np.mean(errors)),
                          "std_test_error": float(np.std(errors, ddof=1)) if len(errors) > 1 else None})
    if aggregate:
        with (root / "aggregate.csv").open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(aggregate[0]))
            writer.writeheader()
            writer.writerows(aggregate)
        for row in aggregate:
            spread = f" ± {row['std_test_error']:.2f}" if row["std_test_error"] is not None else " (std: n/a)"
            print(f"{row['architecture']:8} {row['method']:8} test error: "
                  f"{row['mean_test_error']:.2f}{spread}% | n={row['runs']}")
    else:
        print("Обучение ещё не завершено: показана история, test-метрик пока нет.")
    return aggregate


In [ ]:
aggregate = plot_results(results_dir)
# Для ранее сохранённого запуска достаточно выполнить импорты и ячейку функций выше:
# aggregate = plot_results(Path("runs/zero-init/ИМЯ_КАТАЛОГА"))
# aggregate = plot_results(results_dir, show_std=True)


## Как сформулировать вывод после полного запуска

1. Сравните test error ZerO, Kaiming и Xavier при одинаковом бюджете обучения.
2. Проверьте кривые loss: обучается ли ZerO и насколько быстро относительно baseline.
3. Посмотрите на stable rank выбранных слоёв: отличаются ли начальные значения и траектории.
4. При нескольких seed сравните стандартное отклонение test error. Один запуск не проверяет воспроизводимость.
5. При необходимости добавьте `identity` и `zeros`, чтобы отдельно проверить роль Hadamard и последствие полного обнуления весов.

Результаты короткой проверки нельзя использовать для вывода о превосходстве одного метода. Этот ноутбук сам по себе также не проверяет обучение без BatchNorm или преимущества после pruning.